In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
import pandas as pd

In [2]:
df=pd.read_csv('./Datasets/heart_disease_uci.csv')

In [3]:
df=df.iloc[:,3:]

In [4]:
df.dropna(inplace=True)

In [5]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df.iloc[:,0:-1],df.iloc[:,-1],test_size=0.2,random_state=42)

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [7]:
ohe=OneHotEncoder()


In [8]:
ohe.fit(X_train[['cp','dataset','restecg']])


OneHotEncoder()

In [9]:
X_train_new=ohe.transform(X_train[['cp','dataset','restecg']]).toarray()

In [10]:
X_test_new=ohe.transform(X_test[['cp','dataset','restecg']]).toarray()

In [11]:
clf1=LogisticRegression()
clf1.fit(X_train_new,y_train)

LogisticRegression()

In [12]:
y_prob=clf1.predict_proba(X_test_new)[:,1]

In [13]:
# Filter to binary classification (classes 0 and 1 only)
mask_train = y_train.isin([0, 1])
mask_test = y_test.isin([0, 1])

# Retrain model on binary data
from sklearn.metrics import roc_curve
clf1_binary = LogisticRegression()
clf1_binary.fit(X_train_new[mask_train], y_train[mask_train])

# Get predictions for ROC curve
y_prob_binary = clf1_binary.predict_proba(X_test_new[mask_test])[:, 1]
fpr, tpr, threshold = roc_curve(y_test[mask_test], y_prob_binary)

In [14]:
import plotly.graph_objects as go
import numpy as np


# Generate a trace for ROC curve
trace0 = go.Scatter(
    x=fpr,
    y=tpr,
    mode='lines',
    name='ROC curve'
)

# Only label every nth point to avoid cluttering
n = 10
indices = np.arange(len(threshold)) % n == 0  # Choose indices where index mod n is 0

trace1 = go.Scatter(
    x=fpr[indices],
    y=tpr[indices],
    mode='markers+text',
    name='Threshold points',
    text=[f"Thr={thr:.2f}" for thr in threshold[indices]],
    textposition='top center'
)


# Diagonal line
trace2 = go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random (Area = 0.5)',
    line=dict(dash='dash')
)

data = [trace0, trace1, trace2]

# Define layout with square aspect ratio
layout = go.Layout(
    title='Receiver Operating Characteristic',
    xaxis=dict(title='False Positive Rate'),
    yaxis=dict(title='True Positive Rate'),
    autosize=False,
    width=800,
    height=800,
    showlegend=False
)

# Define figure and add data
fig = go.Figure(data=data, layout=layout)

# Show figure
fig.show()
